# Change claim benefit

### Initiating DB connection, and create a connection function.

In [4]:
# Install required libraries (run this once)
!pip install paramiko pymysql sshtunnel
!pip install python-dotenv

# Import required libraries
import paramiko
import pymysql
from sshtunnel import SSHTunnelForwarder

# Import required library for reading .env file
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Configuration for SSH
SSH_HOST = os.getenv("SSH_HOST")  # Read from .env file
SSH_PORT = int(os.getenv("SSH_PORT", 38123))  # Default SSH port if not specified
SSH_USER = os.getenv("SSH_USER")  # Read from .env file
SSH_KEY = os.getenv("SSH_KEY")  # Read from .env file
SSH_PASSPHRASE = os.getenv("SSH_PASSPHRASE")  # Read from .env file

# Configuration for MySQL
MYSQL_HOST = os.getenv("MYSQL_HOST")  # Read from .env file
MYSQL_PORT = int(os.getenv("MYSQL_PORT", 3306))  # Default MySQL port if not specified
MYSQL_USER = os.getenv("MYSQL_USER")  # Read from .env file
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")  # Read from .env file
MYSQL_DB = "claim_service_development"
def create_db_connection():
    """
    Creates and returns a MySQL database connection through an SSH tunnel.
    
    Returns:
        connection: A MySQL database connection object.
    """
    try:
        # Create an SSH tunnel
        tunnel = SSHTunnelForwarder(
            (SSH_HOST, SSH_PORT),
            ssh_username=SSH_USER,
            ssh_pkey=SSH_KEY,
            ssh_private_key_password=SSH_PASSPHRASE,
            remote_bind_address=(MYSQL_HOST, MYSQL_PORT)
        )
        tunnel.start()
        
        # Connect to MySQL through the SSH tunnel
        connection = pymysql.connect(
            host='127.0.0.1',  # Localhost because of the tunnel
            port=tunnel.local_bind_port,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            db=MYSQL_DB
        )
        
        return connection, tunnel
    
    except Exception as e:
        print(f"An error occurred while creating the connection: {e}")
        return None, None


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


### Create a rollback DML from claim number.

In [5]:
def create_rollback(connection, query, params=None):
    """
    Executes an SQL query using an existing database connection.
    
    Args:
        connection: A MySQL database connection object.
        query (str): The SQL query to execute.
        params (tuple, optional): Parameters to safely inject into the query.
    
    Returns:
        list: A list of tuples (or dictionaries) containing the query results.
    """
    try:
        # Create a cursor object (use DictCursor for dictionary results)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        # Execute the SQL query with parameters
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        
        # Fetch the results
        results = cursor.fetchall()
        
        # Close the cursor (but leave the connection open)
        cursor.close()
        
        return results
    
    except Exception as e:
        print(f"An error occurred while executing the query: {e}")
        return None

# Create a connection
connection, tunnel = create_db_connection()

if connection:
    try:
        # Example query
        query = """
            SELECT c.id, c.number, c.benefit_code, c.benefit_name
            FROM claims c 
            WHERE c.number = %s;
        """
        
        # Get user input and validate it
        user_input = input("Please enter the claim number: ").strip()
        # conditional = "FL-73"
        if not user_input:
            print("Error: Claim number be empty.")
        else:
            # Convert params to tuple correctly
            params = (user_input,)
            # params = (conditional,)
            # Execute the query
            results = create_rollback(connection, query, params)  # ✅ Fixed function call
            
            if results:
                print("Query Results:")
                for row in results:
                    print(row)
                    
                    # Extract values from the result
                    claim_number = row['number']
                    benefit_code = row['benefit_code']
                    benefit_name = row['benefit_name']

                    # Generate rollback
                    print("DML Rollback:")
                    print(f"UPDATE claim_service_development.claims c SET c.updated_at = NOW(), c.benefit_code = '{benefit_code}', c.benefit_name = '{benefit_name}' WHERE c.number = '{claim_number}';")

            else:
                print("No results returned.")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    finally:
        # Close the connection and tunnel when done
        if connection:
            connection.close()
        if tunnel:
            tunnel.stop()
else:
    print("Failed to establish a database connection.")

Query Results:
{'id': 21401, 'number': 'C-20250206-TTRINKAII001-9AV1Q6', 'benefit_code': 'TRIN-MNC-KKAKKK-I3WJ6', 'benefit_name': 'Kompensasi Ketidaknyamanan Atas Keterlambatan Keberangkatan Kereta'}
DML Rollback:
UPDATE claim_service_development.claims c SET c.updated_at = NOW(), c.benefit_code = 'TRIN-MNC-KKAKKK-I3WJ6', c.benefit_name = 'Kompensasi Ketidaknyamanan Atas Keterlambatan Keberangkatan Kereta' WHERE c.number = 'C-20250206-TTRINKAII001-9AV1Q6';


### Update claim number with new benefit code and benefit name.

In [12]:
def update_claim_record(connection, benefit_code, benefit_name, claim_number):
    """
    Updates the claims table with new benefit_code and benefit_name for a given claim_number.
    
    Args:
        connection: A MySQL database connection object.
        benefit_code (str): The new benefit code.
        benefit_name (str): The new benefit name.
        claim_number (str): The claim number to update.
    
    Returns:
        bool: True if the update was successful, False otherwise.
    """
    query = """
        UPDATE claim_service_development.claims c 
        SET c.updated_at = NOW(), 
            c.benefit_code = %s, 
            c.benefit_name = %s 
        WHERE c.number = %s;
    """
    
    try:
        with connection.cursor() as cursor:
            # Show confirmation message with details
            print(f"\nYou are about to update claim number {claim_number}")
            print(f"New benefit code: {benefit_code}")
            print(f"New benefit name: {benefit_name}")
            print("Updating with this DML: ")
            print(f"UPDATE claim_service_development.claims c SET c.updated_at = NOW(), c.benefit_code = '{benefit_code}', c.benefit_name = '{benefit_name}' WHERE c.number = '{claim_number}';")
            confirmation = input("\nDo you want to proceed with this update? (y/n): ").strip().lower()
            
            if confirmation == 'y':
                cursor.execute(query, (benefit_code, benefit_name, claim_number))
                connection.commit()  # Commit the update
                print("Record updated successfully.")
                return True
            else:
                print("Update cancelled by user.")
                return False
    except Exception as e:
        print(f"An error occurred while updating the record: {e}")
        return False

# Create a connection
connection, tunnel = create_db_connection()

if connection:
    try:
        # Get user inputs
        benefit_code = input("Enter new benefit code: ").strip()
        benefit_name = input("Enter new benefit name: ").strip()
        claim_number = input("Enter claim number to update: ").strip()
        
        # Validate inputs
        if not benefit_code or not benefit_name or not claim_number:
            print("Error: All fields must be filled.")
        else:
            success = update_claim_record(connection, benefit_code, benefit_name, claim_number)
            if success:
                print("Update successful.")
            else:
                print("Update failed.")
    
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    finally:
        # Close the connection and tunnel when done
        if connection:
            connection.close()
        if tunnel:
            tunnel.stop()
else:
    print("Failed to establish a database connection.")



You are about to update claim number C-20250206-TTRINKAII001-9AV1Q6
New benefit code: TRIN-MNC-KKAKKK-I3WJ6
New benefit name: Kompensasi Ketidaknyamanan Atas Keterlambatan Keberangkatan Kereta
Updating with this DML: 
UPDATE claim_service_development.claims c SET c.updated_at = NOW(), c.benefit_code = 'TRIN-MNC-KKAKKK-I3WJ6', c.benefit_name = 'Kompensasi Ketidaknyamanan Atas Keterlambatan Keberangkatan Kereta' WHERE c.number = 'C-20250206-TTRINKAII001-9AV1Q6';
Record updated successfully.
Update successful.
